# RelCheck Plan A — COCO Synthetic Evaluation

## Architecture

| Stage | Name | Method | Input → Output |
|-------|------|--------|----------------|
| 1 | Triple Extractor | Llama-3.3-70B | caption → [{subj, rel, obj, type, span}] |
| 2 | Entity Normalizer | Rules-only (spaCy + synonyms) | entity text → normalized form |
| 2b | Entity Grounder | GroundingDINO | image + entities → bounding boxes |
| 3 | Question Generator | Templates (spatial) + LLM (action) | triple → yes/no questions |
| 3b | VQA Executor | Maverick VLM (logprobs) | image + questions → yes_ratios |
| 4 | Verdict Decision | Deterministic rules | geometry + VQA → supported/unsupported/uncertain |
| 5 | Correction Planner | Llama-3.3-70B | caption + triple + verdict → operation + target_span |
| 6 | Minimal Editor | String ops + LLM fallback | caption + plan → edited caption |
| 7 | Safety Check | Rules + LLM | original + edited → approved/rejected |

## Evaluation: Kim-style Synthetic Corruption on COCO val2014

1. Take human GT captions (known correct) from COCO val2014
2. Use LLM to corrupt exactly one relation → corrupted caption
3. Run full pipeline (Stages 1-7) on corrupted caption + image
4. Measure: detection recall, correction quality (BLEU/ROUGE vs GT), no-change accuracy

## Cells

| # | What | Run time |
|---|------|----------|
| 1 | Setup + constants + API helpers | ~2 min |
| 2 | Load GroundingDINO + COCO data | ~5 min |
| 3 | Stage functions (1-7) | instant |
| 4 | COCO synthetic evaluation | ~40-60 min for 100 images |


In [ ]:
# ============================================================
# Cell 1 — Setup + Constants + API Helpers
# ============================================================
!pip install -q together gdown transformers pillow torch torchvision accelerate
!pip install -q spacy pycocotools rouge-score nltk
!python -m spacy download en_core_web_sm -q

import os, json, base64, re, time, random, math
from io import BytesIO
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw
import torch
from together import Together
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import spacy
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from google.colab import drive
drive.mount("/content/drive")

# ── Settings ──────────────────────────────────────────────
TOGETHER_API_KEY = ""   # <-- paste your Together.ai key
N_SYNTH_IMAGES   = 100  # images for COCO synthetic eval
RANDOM_SEED      = 42

# Drive paths
SAVE_DIR         = "/content/drive/MyDrive/RelCheck_Data/plan_a_run"

# COCO paths (for synthetic evaluation)
COCO_DIR         = "/content/coco"
COCO_IMG_DIR     = f"{COCO_DIR}/val2014"
COCO_ANN_FILE    = f"{COCO_DIR}/annotations/captions_val2014.json"

# Model IDs
VLM_MODEL  = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"
LLM_MODEL  = "meta-llama/Llama-3.3-70B-Instruct-Turbo"
GDINO_ID   = "IDEA-Research/grounding-dino-tiny"

# Detection thresholds (Stage 4 — deterministic rules)
GDINO_BOX_THRESHOLD  = 0.25
GDINO_TEXT_THRESHOLD  = 0.20
SPATIAL_DEADZONE     = 0.08
YES_SUPPORTED        = 0.65   # yes_ratio >= this -> supported
YES_UNSUPPORTED      = 0.40   # yes_ratio < this  -> unsupported
# Between 0.40 and 0.65 -> uncertain

os.makedirs(SAVE_DIR, exist_ok=True)
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
client = Together(api_key=TOGETHER_API_KEY)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(RANDOM_SEED)
nlp = spacy.load("en_core_web_sm")

# ── BLEU/ROUGE helpers ────────────────────────────────────
_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
_smooth = SmoothingFunction().method1

def compute_bleu(reference: str, hypothesis: str) -> dict:
    """Compute BLEU-1..4 between two strings."""
    ref_tokens = nltk.word_tokenize(reference.lower())
    hyp_tokens = nltk.word_tokenize(hypothesis.lower())
    if not hyp_tokens:
        return {"bleu1": 0, "bleu2": 0, "bleu3": 0, "bleu4": 0}
    return {
        f"bleu{n}": sentence_bleu(
            [ref_tokens], hyp_tokens,
            weights=tuple([1/n]*n + [0]*(4-n)),
            smoothing_function=_smooth,
        )
        for n in range(1, 5)
    }

def compute_rouge(reference: str, hypothesis: str) -> dict:
    """Compute ROUGE-1, ROUGE-2, ROUGE-L F1 between two strings."""
    scores = _rouge.score(reference, hypothesis)
    return {k: scores[k].fmeasure for k in ["rouge1", "rouge2", "rougeL"]}

# ── Image encoding ────────────────────────────────────────
def encode_b64(img: Image.Image) -> str:
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

# ── LLM call (text-only, returns parsed JSON) ─────────────
def llm_call(prompt: str, model=LLM_MODEL, max_tokens=512, retries=3):
    """Send text prompt, parse JSON response. Returns dict or None."""
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0, max_tokens=max_tokens,
            )
            raw = resp.choices[0].message.content.strip()
            # Strip markdown code fences
            raw = re.sub(r"^```(?:json)?\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            return json.loads(raw)
        except json.JSONDecodeError:
            try:
                m = re.search(r'(\{.*\}|\[.*\])', raw, re.DOTALL)
                if m:
                    return json.loads(m.group(1))
            except Exception:
                pass
            if attempt < retries - 1:
                time.sleep(1)
            else:
                print(f"  [llm] JSON parse failed. Raw: {raw[:200]}")
                return None
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"  [llm] API error: {e}")
                return None

# ── LLM call (text-only, returns RAW string, no JSON parsing) ──
def llm_call_raw(prompt: str, model=LLM_MODEL, max_tokens=512, retries=3):
    """Send text prompt, return raw text response (no JSON parsing)."""
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0, max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"  [llm_raw] API error: {e}")
                return None

# ── VLM logprob (yes/no with confidence) ──────────────────
def vlm_logprob(img: Image.Image, question: str, model=VLM_MODEL, retries=3):
    """Ask yes/no question with logprobs. Returns (yes_ratio, confidence)."""
    b64 = encode_b64(img)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                    {"type": "text", "text": question + " Answer with exactly one word: yes or no."},
                ]}],
                temperature=0.0, max_tokens=1, logprobs=True,
            )
            choice = resp.choices[0]
            answer = (choice.message.content or "").strip().lower()
            lp_list = getattr(choice.logprobs, "content", None) or []
            top_lps = lp_list[0].top_logprobs if lp_list else []
            probs = {e.token.strip().lower(): np.exp(e.logprob) for e in top_lps}
            yes_p = sum(v for k, v in probs.items() if k.startswith("y"))
            no_p  = sum(v for k, v in probs.items() if k.startswith("n"))
            total = yes_p + no_p
            if total < 0.05:
                yes_p = 1.0 if answer.startswith("y") else 0.0
                total = 1.0
            return (yes_p / total, min(total, 1.0))
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"  [logprob] error: {e}")
                return (None, None)

# ── VLM free-text call (for contrastive questions) ────────
def vlm_freetext(img: Image.Image, prompt: str, model=VLM_MODEL, max_tokens=50, retries=3):
    """Send image + text, return raw string response."""
    b64 = encode_b64(img)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                    {"type": "text", "text": prompt},
                ]}],
                temperature=0.0, max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"  [vlm_text] error: {e}")
                return None

# ── Checkpoint helpers ────────────────────────────────────
def save_ckpt(data, path):
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)

def load_ckpt(path):
    return json.load(open(path)) if os.path.exists(path) else None

print(f"Device: {DEVICE}")
print(f"Save:   {SAVE_DIR}")
print("Setup complete.")

In [ ]:
# ============================================================
# Cell 2 — Load GroundingDINO + COCO Data
# ============================================================
import subprocess

# ── GroundingDINO ─────────────────────────────────────────
print("Loading GroundingDINO...")
gdino_proc  = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
gdino_model.eval()
print("  GroundingDINO ready.")

# ── COCO val2014 (unzip from Drive) ──────────────────────
print("\nSetting up COCO val2014...")
os.makedirs(COCO_DIR, exist_ok=True)
os.makedirs(f"{COCO_DIR}/annotations", exist_ok=True)

# Unzip images from Drive zip (skips if already extracted)
if not os.path.isdir(COCO_IMG_DIR) or len(os.listdir(COCO_IMG_DIR)) < 100:
    assert os.path.exists(COCO_IMG_ZIP), (
        f"val2014.zip not found at {COCO_IMG_ZIP}\n"
        "Download from http://images.cocodataset.org/zips/val2014.zip "
        "and upload to MyDrive/RelCheck_Data/coco_zips/ on Drive."
    )
    print("  Unzipping val2014.zip from Drive (~6.2 GB, takes a few minutes)...")
    subprocess.run(["unzip", "-q", COCO_IMG_ZIP, "-d", COCO_DIR], check=True)
    print(f"  Images ready ({len(os.listdir(COCO_IMG_DIR))} files).")
else:
    print(f"  COCO images already present ({len(os.listdir(COCO_IMG_DIR))} files).")

# Unzip annotations from Drive zip (skips if already extracted)
if not os.path.exists(COCO_ANN_FILE):
    assert os.path.exists(COCO_ANN_ZIP), (
        f"annotations zip not found at {COCO_ANN_ZIP}\n"
        "Download from http://images.cocodataset.org/annotations/annotations_trainval2014.zip "
        "and upload to MyDrive/RelCheck_Data/coco_zips/ on Drive."
    )
    print("  Unzipping annotations from Drive...")
    subprocess.run(["unzip", "-q", "-o", COCO_ANN_ZIP, "-d", COCO_DIR], check=True)
    print("  Annotations ready.")
else:
    print("  COCO annotations already present.")

# ── Load COCO captions ───────────────────────────────────
from pycocotools.coco import COCO
coco_caps = COCO(COCO_ANN_FILE)

coco_img_ids = coco_caps.getImgIds()
random.seed(RANDOM_SEED)
random.shuffle(coco_img_ids)

# Select images with >= 3 GT captions (2x buffer for both tracks + skips)
coco_data = {}
for cid in coco_img_ids:
    if len(coco_data) >= N_SYNTH_IMAGES * 2:
        break
    ann_ids = coco_caps.getAnnIds(imgIds=cid)
    anns = coco_caps.loadAnns(ann_ids)
    gt_caps = [a["caption"].strip() for a in anns if a["caption"].strip()]
    if len(gt_caps) < 3:
        continue
    img_info = coco_caps.loadImgs(cid)[0]
    img_path = os.path.join(COCO_IMG_DIR, img_info["file_name"])
    if not os.path.exists(img_path):
        continue
    coco_data[cid] = {
        "path": img_path,
        "file_name": img_info["file_name"],
        "gt_captions": gt_caps,
    }

print(f"\n  COCO images loaded: {len(coco_data)}")
print(f"  Avg GT captions/image: {np.mean([len(v['gt_captions']) for v in coco_data.values()]):.1f}")

for cid in list(coco_data.keys())[:3]:
    d = coco_data[cid]
    print(f"  COCO {cid}: {d['gt_captions'][0][:80]}...")


In [ ]:
# ============================================================
# Cell 4 — ALL STAGE FUNCTIONS (Stages 1-7)
# ============================================================

# ─────────────────────────────────────────────────────────
# STAGE 1: Triple Extraction (LLM)
# ─────────────────────────────────────────────────────────
TRIPLE_EXTRACT_PROMPT = """Extract all (subject, relation, object) triples from the caption.

Caption: {caption}

For each triple, identify:
1. subject: entity performing/experiencing action
2. relation: verb or spatial preposition (e.g., "is wearing", "to the left of")
3. object: entity being acted upon
4. relation_type: classify as SPATIAL (left/right/above/below/on/under/near) or ACTION (verb) or ATTRIBUTE (is/has/color) or COUNT (number/amount) or OTHER
5. sentence_text: the full sentence containing this triple
6. supporting_span: the exact substring in caption that contains subject + relation + object

Return JSON array:
[
  {
    "subject": "...",
    "relation": "...",
    "object": "...",
    "relation_type": "SPATIAL|ACTION|ATTRIBUTE|COUNT|OTHER",
    "sentence_text": "...",
    "supporting_span": "..."
  },
  ...
]

Return JSON only. No markdown fences."""

def stage1_extract_triples(caption: str) -> list:
    """
    Stage 1: Extract triples from caption using Llama-3.3-70B.
    Returns list of triple dicts with subject, relation, object, relation_type, sentence_text, supporting_span.
    """
    prompt = TRIPLE_EXTRACT_PROMPT.format(caption=caption)
    result = llm_call(prompt, max_tokens=1024)
    
    if result is None or not isinstance(result, list):
        print(f"  [S1] Triple extraction failed for caption: {caption[:60]}")
        return []
    
    # Validate each triple has required fields
    triples = []
    for t in result:
        if all(k in t for k in ["subject", "relation", "object", "relation_type"]):
            triples.append({
                "subject": t["subject"].strip(),
                "relation": t["relation"].strip(),
                "object": t["object"].strip(),
                "relation_type": t["relation_type"].strip().upper(),
                "sentence_text": t.get("sentence_text", "").strip(),
                "supporting_span": t.get("supporting_span", "").strip(),
            })
    
    print(f"  [S1] Extracted {len(triples)} triples from caption: {caption[:50]}")
    return triples


# ─────────────────────────────────────────────────────────
# STAGE 2: Entity Normalization (Rules Only - No API)
# ─────────────────────────────────────────────────────────

SYNONYM_MAP = {
    "man": ["man", "male", "boy", "gentleman", "person", "guy"],
    "woman": ["woman", "female", "girl", "lady", "person"],
    "person": ["person", "man", "woman", "people", "human"],
    "child": ["child", "kid", "boy", "girl"],
    "dog": ["dog", "puppy", "canine"],
    "cat": ["cat", "kitten", "feline"],
    "bird": ["bird", "eagle", "sparrow"],
    "car": ["car", "vehicle", "automobile", "sedan", "truck"],
    "bike": ["bike", "bicycle", "motorcycle", "cycle"],
    "motorcycle": ["motorcycle", "bike", "cycle"],
    "table": ["table", "desk"],
    "chair": ["chair", "seat"],
    "book": ["book", "magazine", "newspaper"],
    "phone": ["phone", "cellphone", "smartphone", "mobile"],
    "bottle": ["bottle", "jar"],
    "cup": ["cup", "mug", "glass"],
    "plate": ["plate", "dish"],
    "hat": ["hat", "cap", "helmet"],
    "shirt": ["shirt", "top", "blouse"],
    "pants": ["pants", "trousers", "jeans"],
    "shoe": ["shoe", "boot", "sneaker"],
    "flower": ["flower", "rose", "daisy"],
    "tree": ["tree", "plant"],
    "grass": ["grass", "lawn"],
    "water": ["water", "lake", "river", "ocean"],
    "sky": ["sky", "clouds"],
}

def stage2_normalize_entity(text: str) -> dict:
    """
    Normalize entity text through 4 layers:
    1. Text cleanup (lowercase, remove articles, punctuation)
    2. spaCy head noun + lemma
    3. Synonym map lookup
    4. Return normalized form + candidates
    """
    # Layer 1: cleanup
    text_clean = text.lower().strip()
    text_clean = re.sub(r'^(a|an|the)\s+', '', text_clean)  # Remove articles
    text_clean = re.sub(r'[,;:\!?]', '', text_clean)  # Remove punctuation
    text_clean = text_clean.strip()
    
    # Layer 2: spaCy head noun + lemma
    doc = nlp(text_clean)
    head_noun = None
    lemma = None
    if doc.text.strip():
        # Find the head noun (rightmost noun in phrase)
        for token in doc:
            if token.pos_ == "NOUN":
                lemma = token.lemma_
                head_noun = token.text
        if not head_noun and doc:
            head_noun = doc[-1].text
    
    # Layer 3: synonym map
    canonical = None
    candidates = []
    if head_noun:
        for canon, syns in SYNONYM_MAP.items():
            if head_noun in [s.lower() for s in syns] or head_noun == canon:
                canonical = canon
                candidates = syns
                break
    
    return {
        "original": text,
        "cleaned": text_clean,
        "head_noun": head_noun,
        "lemma": lemma,
        "canonical": canonical,
        "synonym_candidates": candidates,
    }

def stage2_match_entity(entity_norm: dict, detections: list) -> list:
    """
    Match normalized entity against detected objects.
    Returns list of matching detection indices (could be multiple due to synonyms).
    """
    if not entity_norm.get("synonym_candidates"):
        return []
    
    candidates = [c.lower() for c in entity_norm["synonym_candidates"]]
    matches = []
    
    for i, det in enumerate(detections):
        det_label = (det.get("label") or "").lower()
        # Exact match or substring match
        if det_label in candidates or any(c in det_label or det_label in c for c in candidates):
            matches.append(i)
    
    return matches


# ─────────────────────────────────────────────────────────
# STAGE 2b: Entity Grounding (GroundingDINO)
# ─────────────────────────────────────────────────────────

def run_gdino(img: Image.Image, queries: list) -> dict:
    """
    Run GroundingDINO to detect objects matching query labels.
    Returns dict: {query: [{"box": [x1,y1,x2,y2], "score": float, "label": str}]}
    """
    results = {}
    
    # Process each query separately to get per-query results
    for query in queries:
        if not query.strip():
            results[query] = []
            continue
        
        inputs = gdino_proc(images=img, text=query, return_tensors="pt").to(DEVICE)
        
        with torch.no_grad():
            outputs = gdino_model(**inputs)
        
        # Post-process: filter by thresholds
        logits = outputs.logits.sigmoid()[0]  # [num_boxes, 1]
        boxes = outputs.pred_boxes[0]  # [num_boxes, 4] in normalized coords (0-1)
        
        # Get scores for the query (if multi-label, take max across labels)
        scores = logits.max(dim=1).values if logits.dim() > 1 else logits
        
        # Filter by text and box thresholds
        mask = scores >= GDINO_TEXT_THRESHOLD
        filtered_boxes = boxes[mask]
        filtered_scores = scores[mask]
        
        # Convert to pixel coordinates (assume image is standard size)
        h, w = img.height, img.width
        boxes_px = []
        for i, (score, box_norm) in enumerate(zip(filtered_scores, filtered_boxes)):
            x1_n, y1_n, x2_n, y2_n = box_norm.tolist()
            x1, y1 = int(x1_n * w), int(y1_n * h)
            x2, y2 = int(x2_n * w), int(y2_n * h)
            
            if score.item() >= GDINO_BOX_THRESHOLD:
                boxes_px.append({
                    "box": [x1, y1, x2, y2],
                    "score": score.item(),
                    "label": query.strip(),
                })
        
        results[query] = boxes_px
    
    return results


# ─────────────────────────────────────────────────────────
# STAGE 3: Verification Question Generation
# ─────────────────────────────────────────────────────────

SPATIAL_OPPOSITES = {
    "left of": "right of",
    "right of": "left of",
    "above": "below",
    "below": "above",
    "on": "under",
    "under": "on",
    "next to": "far from",
    "near": "far from",
}

def _spatial_questions(subj: str, rel: str, obj: str) -> dict:
    """Generate yes/no template questions for spatial relations (no API)."""
    rel_lower = rel.lower()
    
    questions = []
    
    # Standard yes question
    questions.append(f"Is the {subj} {rel} the {obj}?")
    
    # Passive/alternate phrasing
    if "left" in rel_lower or "right" in rel_lower:
        if "left" in rel_lower:
            questions.append(f"Is the {obj} to the right of the {subj}?")
        else:
            questions.append(f"Is the {obj} to the left of the {subj}?")
    elif "above" in rel_lower or "below" in rel_lower:
        if "above" in rel_lower:
            questions.append(f"Is the {obj} below the {subj}?")
        else:
            questions.append(f"Is the {obj} above the {subj}?")
    
    # Opposite relation (contrastive)
    opposite = SPATIAL_OPPOSITES.get(rel_lower, None)
    if opposite:
        opposite_q = f"Is the {subj} {opposite} the {obj}?"
        return {
            "yes_no_questions": questions,
            "contrastive_question": opposite_q,
        }
    
    return {
        "yes_no_questions": questions,
        "contrastive_question": None,
    }

ACTION_ATTR_QUESTIONS_PROMPT = """Generate 2-3 yes/no verification questions for this triple.

Subject: {subject}
Relation: {relation}
Object: {object}

Generate questions that test whether the relation holds in an image. Vary phrasing (active/passive).
For action relations: "Is the X [relation] the Y?", "Is the Y being [relation] by the X?", etc.
For attributes: "Does the X have/are/is [relation]?", etc.

Return JSON:
{{
  "yes_no_questions": ["question 1", "question 2", "question 3"],
  "contrastive_question": "Is the X [opposite relation]?" (optional, can be null)
}}

Return JSON only."""

def stage3_generate_questions(triple: dict) -> dict:
    """
    Generate yes/no questions for verification.
    For SPATIAL: use templates.
    For ACTION/ATTRIBUTE: use LLM.
    Returns dict with yes_no_questions list and optional contrastive_question.
    """
    subj = triple.get("subject", "")
    rel = triple.get("relation", "")
    obj = triple.get("object", "")
    rel_type = triple.get("relation_type", "").upper()
    
    if rel_type == "SPATIAL":
        return _spatial_questions(subj, rel, obj)
    else:
        # ACTION or ATTRIBUTE: use LLM
        prompt = ACTION_ATTR_QUESTIONS_PROMPT.format(
            subject=subj,
            relation=rel,
            object=obj,
        )
        result = llm_call(prompt, max_tokens=256)
        
        if result and "yes_no_questions" in result:
            return {
                "yes_no_questions": result.get("yes_no_questions", []),
                "contrastive_question": result.get("contrastive_question"),
            }
        
        # Fallback if LLM fails
        return {
            "yes_no_questions": [f"Is the {subj} {rel} the {obj}?"],
            "contrastive_question": None,
        }

def stage3_run_vqa(img: Image.Image, questions: dict) -> dict:
    """
    Run VQA questions on the image using Maverick VLM.
    Returns dict with yes_no_answers (list of yes_ratio, confidence tuples) and contrastive_answer.
    """
    yes_no_qs = questions.get("yes_no_questions", [])
    contrastive_q = questions.get("contrastive_question")
    
    # Run yes/no questions with logprobs
    yes_no_answers = []
    for q in yes_no_qs:
        yes_ratio, confidence = vlm_logprob(img, q)
        if yes_ratio is not None:
            yes_no_answers.append({
                "question": q,
                "yes_ratio": yes_ratio,
                "confidence": confidence,
            })
    
    # Run contrastive question if present
    contrastive_answer = None
    if contrastive_q:
        text = vlm_freetext(img, contrastive_q)
        contrastive_answer = {
            "question": contrastive_q,
            "answer": text,
        }
    
    return {
        "yes_no_answers": yes_no_answers,
        "contrastive_answer": contrastive_answer,
    }


# ─────────────────────────────────────────────────────────
# SPATIAL GEOMETRY VERIFICATION (Rules Only)
# ─────────────────────────────────────────────────────────

def centroid(box: list) -> tuple:
    """Compute centroid of bounding box [x1, y1, x2, y2]."""
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

def horizontal_overlap(b1: list, b2: list) -> float:
    """Compute horizontal overlap ratio between two boxes."""
    x1_min, _, x1_max, _ = b1
    x2_min, _, x2_max, _ = b2
    
    overlap_min = max(x1_min, x2_min)
    overlap_max = min(x1_max, x2_max)
    overlap = max(0, overlap_max - overlap_min)
    
    width1 = x1_max - x1_min
    width2 = x2_max - x2_min
    max_width = max(width1, width2)
    
    return overlap / max_width if max_width > 0 else 0

def verify_spatial_geometry(rel: str, subj_box: list, obj_box: list) -> dict:
    """
    Verify spatial relation using bounding box geometry.
    Returns dict with verdict (True/False/None), evidence, and correct_relation suggestion.
    """
    rel_lower = rel.lower().strip()
    
    subj_cx, subj_cy = centroid(subj_box)
    obj_cx, obj_cy = centroid(obj_box)
    h_overlap = horizontal_overlap(subj_box, obj_box)
    
    # Heuristic rules for common spatial relations
    verdict = None
    evidence = ""
    correct_relation = None
    
    if rel_lower in ["left of", "to the left of"]:
        if subj_cx < obj_cx:
            verdict = True
            evidence = f"Subject centroid ({subj_cx:.0f}) < object centroid ({obj_cx:.0f})"
        else:
            verdict = False
            correct_relation = "to the right of"
            evidence = f"Subject centroid ({subj_cx:.0f}) >= object centroid ({obj_cx:.0f})"
    
    elif rel_lower in ["right of", "to the right of"]:
        if subj_cx > obj_cx:
            verdict = True
            evidence = f"Subject centroid ({subj_cx:.0f}) > object centroid ({obj_cx:.0f})"
        else:
            verdict = False
            correct_relation = "to the left of"
            evidence = f"Subject centroid ({subj_cx:.0f}) <= object centroid ({obj_cx:.0f})"
    
    elif rel_lower in ["above", "on top of"]:
        if subj_cy < obj_cy:
            verdict = True
            evidence = f"Subject centroid Y ({subj_cy:.0f}) < object centroid Y ({obj_cy:.0f})"
        else:
            verdict = False
            correct_relation = "below"
            evidence = f"Subject centroid Y ({subj_cy:.0f}) >= object centroid Y ({obj_cy:.0f})"
    
    elif rel_lower in ["below", "underneath"]:
        if subj_cy > obj_cy:
            verdict = True
            evidence = f"Subject centroid Y ({subj_cy:.0f}) > object centroid Y ({obj_cy:.0f})"
        else:
            verdict = False
            correct_relation = "above"
            evidence = f"Subject centroid Y ({subj_cy:.0f}) <= object centroid Y ({obj_cy:.0f})"
    
    elif rel_lower in ["on", "on top of"]:
        # Strict: subject must be above and have significant horizontal overlap
        if subj_cy < obj_cy and h_overlap > 0.3:
            verdict = True
            evidence = f"Subject above (Y:{subj_cy:.0f} < {obj_cy:.0f}) + overlap:{h_overlap:.2f}"
        else:
            verdict = False
            evidence = f"Insufficient overlap or wrong vertical position"
    
    elif rel_lower in ["under", "underneath"]:
        if subj_cy > obj_cy and h_overlap > 0.3:
            verdict = True
            evidence = f"Subject below (Y:{subj_cy:.0f} > {obj_cy:.0f}) + overlap:{h_overlap:.2f}"
        else:
            verdict = False
            evidence = f"Insufficient overlap or wrong vertical position"
    
    elif rel_lower in ["next to", "beside", "near"]:
        # Loose: nearby horizontally or vertically
        dist = math.sqrt((subj_cx - obj_cx)**2 + (subj_cy - obj_cy)**2)
        if dist < 200:  # Heuristic pixel distance threshold
            verdict = True
            evidence = f"Proximity distance: {dist:.0f}px"
        else:
            verdict = False
            correct_relation = "far from"
            evidence = f"Distance too large: {dist:.0f}px"
    
    # If relation is unrecognized, return None (will trigger VQA fallback)
    if verdict is None:
        evidence = f"Unsupported spatial relation type: {rel}"
    
    return {
        "verdict": verdict,
        "evidence": evidence,
        "correct_relation": correct_relation,
    }


# ─────────────────────────────────────────────────────────
# STAGE 4: Verdict Decision (Rules - No API)
# ─────────────────────────────────────────────────────────

def stage4_decide(triple: dict, geometry_result: dict, vqa_result: dict) -> dict:
    """
    Decide verdict based on geometry + VQA results.
    Rules:
    - Spatial with geometry verdict not None -> use geometry (confidence 0.95)
    - VQA: avg yes_ratio >= YES_SUPPORTED -> supported
    - VQA: avg yes_ratio < YES_UNSUPPORTED -> unsupported
    - Middle zone: check contrastive question
    """
    rel_type = triple.get("relation_type", "").upper()
    
    # Rule 1: If we have geometry verdict for spatial, use it
    if rel_type == "SPATIAL" and geometry_result.get("verdict") is not None:
        return {
            "verdict": "supported" if geometry_result["verdict"] else "hallucinated",
            "confidence": 0.95,
            "evidence_used": "spatial_geometry",
            "geometry_details": geometry_result,
            "correct_relation": geometry_result.get("correct_relation"),
        }
    
    # Rule 2: Use VQA results
    vqa_answers = vqa_result.get("yes_no_answers", [])
    if not vqa_answers:
        return {
            "verdict": "uncertain",
            "confidence": 0.0,
            "evidence_used": "none",
            "reason": "No VQA answers collected",
        }
    
    # Compute average yes_ratio
    yes_ratios = [a.get("yes_ratio", 0.5) for a in vqa_answers]
    avg_yes_ratio = np.mean(yes_ratios)
    
    # Rule 3: High yes_ratio -> supported
    if avg_yes_ratio >= YES_SUPPORTED:
        return {
            "verdict": "supported",
            "confidence": avg_yes_ratio,
            "evidence_used": "vqa",
            "yes_ratios": yes_ratios,
        }
    
    # Rule 4: Low yes_ratio -> hallucinated
    if avg_yes_ratio < YES_UNSUPPORTED:
        return {
            "verdict": "hallucinated",
            "confidence": 1.0 - avg_yes_ratio,
            "evidence_used": "vqa",
            "yes_ratios": yes_ratios,
        }
    
    # Rule 5: Uncertain zone [0.40, 0.65) -> check contrastive
    contrastive = vqa_result.get("contrastive_answer")
    if contrastive:
        answer = (contrastive.get("answer") or "").lower()
        if "yes" in answer or "true" in answer:
            return {
                "verdict": "supported",
                "confidence": 0.55,
                "evidence_used": "vqa_contrastive",
                "reason": "Contrastive tipped toward support",
            }
        else:
            return {
                "verdict": "hallucinated",
                "confidence": 0.55,
                "evidence_used": "vqa_contrastive",
                "reason": "Contrastive tipped toward hallucination",
            }
    
    # No contrastive to break tie
    return {
        "verdict": "uncertain",
        "confidence": 0.5,
        "evidence_used": "vqa_uncertain_zone",
        "yes_ratios": yes_ratios,
        "reason": f"Yes ratio {avg_yes_ratio:.2f} in uncertain zone [0.40, 0.65)",
    }


# ─────────────────────────────────────────────────────────
# STAGE 5: Correction Planning (LLM)
# ─────────────────────────────────────────────────────────

PLAN_CORRECTION_PROMPT = """Plan a correction for this hallucinated triple.

Caption: {caption}
Triple: subject='{subject}', relation='{relation}', object='{object}'
Hallucination type: {reason}

The triple is hallucinated. Choose ONE correction operation:
1. "delete_span" — completely remove the phrase from caption
2. "replace_relation" — replace the relation with a correct one (provide replacement_text)
3. "replace_object" — replace the object with correct one (provide replacement_text)
4. "abstain" — too risky, keep original

Return JSON:
{{
  "operation": "delete_span|replace_relation|replace_object|abstain",
  "target_span": "the exact phrase to edit (substring of caption)",
  "replacement_text": "text to insert (null for delete)",
  "reason": "why this operation is safe"
}}

Return JSON only."""

def stage5_plan_correction(caption: str, triple: dict, verdict: dict) -> dict:
    """
    Plan correction operation using LLM.
    Returns dict with operation, target_span, replacement_text, reason.
    """
    subj = triple.get("subject", "")
    rel = triple.get("relation", "")
    obj = triple.get("object", "")
    reason = verdict.get("reason", verdict.get("evidence_used", "hallucinated"))
    
    prompt = PLAN_CORRECTION_PROMPT.format(
        caption=caption,
        subject=subj,
        relation=rel,
        object=obj,
        reason=reason,
    )
    
    result = llm_call(prompt, max_tokens=256)
    
    if result and "operation" in result:
        return {
            "operation": result.get("operation", "abstain"),
            "target_span": result.get("target_span", ""),
            "replacement_text": result.get("replacement_text"),
            "reason": result.get("reason", ""),
        }
    
    # Fallback: try to delete the supporting span if available
    supporting_span = triple.get("supporting_span", "")
    if supporting_span and supporting_span in caption:
        return {
            "operation": "delete_span",
            "target_span": supporting_span,
            "replacement_text": None,
            "reason": "Fallback: delete supporting span",
        }
    
    # Final fallback: abstain
    return {
        "operation": "abstain",
        "target_span": "",
        "replacement_text": None,
        "reason": "Could not plan safe correction",
    }


# ─────────────────────────────────────────────────────────
# STAGE 6: Minimal Local Editor (LLM)
# ─────────────────────────────────────────────────────────

MINIMAL_EDIT_PROMPT = """Edit the caption to fix a hallucination. Edit ONLY the target span.

Original caption: {caption}
Target span to edit: '{target_span}'
Operation: {operation}
Replacement text: {replacement_text}

Rules:
1. Replace or delete ONLY the target span
2. Preserve all other text exactly
3. Maintain grammar and punctuation
4. Output: {{"revised_sentence": "...", "revised_caption": "..."}}

Return JSON only."""

def stage6_minimal_edit(caption: str, triple: dict, plan: dict) -> dict:
    """
    Edit caption using minimal local edits.
    Try string replacement first (rule-based), then LLM fallback.
    Returns dict with revised_sentence, revised_caption.
    """
    operation = plan.get("operation", "abstain")
    target_span = plan.get("target_span", "")
    replacement = plan.get("replacement_text")
    
    # Rule-based approach first
    if operation == "delete_span" and target_span in caption:
        revised = caption.replace(target_span, "").strip()
        # Clean up double spaces
        revised = re.sub(r'\s+', ' ', revised)
        return {
            "operation": operation,
            "revised_caption": revised,
            "revised_sentence": "",
            "method": "rule_based_delete",
        }
    
    if operation == "replace_relation" and target_span in caption:
        if replacement:
            revised = caption.replace(target_span, replacement)
            revised = re.sub(r'\s+', ' ', revised).strip()
            return {
                "operation": operation,
                "revised_caption": revised,
                "revised_sentence": "",
                "method": "rule_based_replace",
            }
    
    # LLM fallback
    prompt = MINIMAL_EDIT_PROMPT.format(
        caption=caption,
        target_span=target_span,
        operation=operation,
        replacement_text=replacement or "null",
    )
    
    result = llm_call(prompt, max_tokens=256)
    
    if result and "revised_caption" in result:
        return {
            "operation": operation,
            "revised_caption": result.get("revised_caption", caption),
            "revised_sentence": result.get("revised_sentence", ""),
            "method": "llm",
        }
    
    # Fallback: keep original
    return {
        "operation": "abstain",
        "revised_caption": caption,
        "revised_sentence": "",
        "method": "fallback_keep_original",
    }


# ─────────────────────────────────────────────────────────
# STAGE 7: Post-Edit Safety Check (LLM)
# ─────────────────────────────────────────────────────────

SAFETY_CHECK_PROMPT = """Evaluate whether this caption edit is safe.

Original caption: {original}
Edited caption: {edited}
Hallucinated triple: subject='{subject}', relation='{relation}', object='{object}'
Operation: {operation}

Check for:
1. Did the edit successfully remove the hallucinated triple?
2. Did the edit introduce any NEW unsupported claims?
3. Is the edited caption grammatically fluent?
4. Did the edit preserve important information?

Return JSON:
{{
  "approved": true|false,
  "reason": "..."
}}

Return JSON only."""

def stage7_safety_check(original: str, edited: str, triple: dict, operation: str) -> dict:
    """
    Safety check: quick rule-based check first, LLM fallback.
    Returns dict with approved (bool) and reason (str).
    """
    # Quick rule-based check
    # If edit lost >60% of text, reject (too aggressive)
    orig_words = len(original.split())
    edit_words = len(edited.split())
    if orig_words > 0 and edit_words < orig_words * 0.4:
        return {
            "approved": False,
            "reason": f"Edit loss too severe: {orig_words} -> {edit_words} words",
            "method": "rule_based",
        }
    
    # If edited caption is empty, reject
    if not edited.strip():
        return {
            "approved": False,
            "reason": "Edited caption is empty",
            "method": "rule_based",
        }
    
    # LLM safety check
    subj = triple.get("subject", "")
    rel = triple.get("relation", "")
    obj = triple.get("object", "")
    
    prompt = SAFETY_CHECK_PROMPT.format(
        original=original,
        edited=edited,
        subject=subj,
        relation=rel,
        object=obj,
        operation=operation,
    )
    
    result = llm_call(prompt, max_tokens=256)
    
    if result and "approved" in result:
        return {
            "approved": result.get("approved", False),
            "reason": result.get("reason", ""),
            "method": "llm",
        }
    
    # Conservative fallback: reject
    return {
        "approved": False,
        "reason": "Safety check inconclusive",
        "method": "fallback_reject",
    }


print("✓ All 7 stages defined (functions ready to call)")

In [ ]:
# ============================================================
# Cell 7 — COCO Synthetic Evaluation (Kim-style Corruption)
# ============================================================
# Method: Take human GT captions (known correct) from COCO val2014,
# inject synthetic relational hallucinations via LLM, run RelCheck
# pipeline, measure detection recall + correction quality (BLEU/ROUGE).
#
# Two test tracks:
#   A) CORRUPTED — GT caption with one relation changed → pipeline should detect + fix
#   B) NO-CHANGE — clean GT caption → pipeline should NOT modify
# ============================================================

SYNTH_CKPT = f"{SAVE_DIR}/coco_synth_results.json"
synth = load_ckpt(SYNTH_CKPT) or {
    "corrupted": [],   # Track A results
    "no_change": [],   # Track B results
    "skipped": 0,
}

# ── Corruption prompt ─────────────────────────────────────
CORRUPT_PROMPT = """You are given a factually correct image caption and one relational triple extracted from it.
Your task: change ONLY the relation in the caption to create a plausible but INCORRECT caption.

Rules:
1. Change exactly ONE relation — the one specified in the triple.
2. The new relation must be plausible (could be true in some image) but wrong for THIS image.
3. Keep all other words, entities, and structure identical.
4. Do NOT add or remove entities. Do NOT paraphrase other parts.
5. The corrupted span should be minimal (1-3 words changed).

Caption: {caption}
Triple to corrupt: ({subject}, {relation}, {object})
The span in the caption containing this relation: "{span}"

Respond with JSON only:
{{
  "corrupted_caption": "...",
  "original_relation": "{relation}",
  "corrupted_relation": "the new (wrong) relation you inserted",
  "original_span": "the exact substring you changed",
  "corrupted_span": "the new substring in the corrupted caption"
}}"""

# ── Helper: run full pipeline on one (image, caption) pair ──
def run_pipeline_on_caption(img, caption):
    """Run Stages 1-7 on a single (image, caption). Returns dict with results."""
    result = {
        "input_caption": caption,
        "corrected_caption": caption,
        "triples": [],
        "verdicts": [],
        "corrections": [],
    }

    # Stage 1: Extract triples
    triples = stage1_extract_triples(caption)
    if not triples:
        return result
    result["triples"] = triples

    # Stage 2: Normalize + ground
    all_entities = set()
    for t in triples:
        t["_subj_norm"] = stage2_normalize_entity(t["subject"])
        t["_obj_norm"] = stage2_normalize_entity(t["object"])
        all_entities.add(t["_subj_norm"]["head_noun"])
        all_entities.add(t["_obj_norm"]["head_noun"])
        all_entities.add(t["subject"].lower())
        all_entities.add(t["object"].lower())

    detections = run_gdino(img, list(all_entities))

    unsupported = []
    for t in triples:
        subj_matches = stage2_match_entity(t["_subj_norm"], detections)
        obj_matches  = stage2_match_entity(t["_obj_norm"], detections)
        subj_box = subj_matches[0]["box"] if subj_matches else None
        obj_box  = obj_matches[0]["box"] if obj_matches else None

        rtype = t.get("relation_type", "other")
        geometry_result = None
        vqa_result = None

        if rtype == "spatial" and subj_box and obj_box:
            geometry_result = verify_spatial_geometry(t["relation"], subj_box, obj_box)
            if geometry_result.get("verdict") is None:
                questions = stage3_generate_questions(t)
                vqa_result = stage3_run_vqa(img, questions)
        else:
            questions = stage3_generate_questions(t)
            vqa_result = stage3_run_vqa(img, questions)

        verdict = stage4_decide(t, geometry_result, vqa_result)
        result["verdicts"].append({"triple": t, "verdict": verdict})

        if verdict.get("verdict") == "unsupported":
            unsupported.append((t, verdict))

    # Stages 5-7: Correct
    corrected = caption
    for t, verdict in unsupported:
        plan = stage5_plan_correction(corrected, t, verdict)
        if plan.get("operation") in ("no_change", "abstain"):
            continue
        edit_result = stage6_minimal_edit(corrected, t, plan)
        candidate = edit_result.get("revised_caption", corrected)
        safety = stage7_safety_check(corrected, candidate, t, plan.get("operation", ""))
        if safety.get("approved", False):
            corrected = candidate
            result["corrections"].append({
                "triple": t,
                "operation": plan.get("operation"),
                "before": caption,
                "after": corrected,
            })

    result["corrected_caption"] = corrected
    return result


# ══════════════════════════════════════════════════════════
# TRACK A: Corrupted captions (inject hallucination → detect + fix)
# ══════════════════════════════════════════════════════════
print("=" * 70)
print("TRACK A: Corrupted GT captions")
print("=" * 70)

already_done_ids = {r["coco_id"] for r in synth["corrupted"]}
coco_ids = [cid for cid in coco_data if cid not in already_done_ids]
n_target_corrupted = max(0, N_SYNTH_IMAGES - len(synth["corrupted"]))

print(f"  Already done: {len(synth['corrupted'])}, remaining: {n_target_corrupted}")

for idx, coco_id in enumerate(coco_ids):
    if len(synth["corrupted"]) >= N_SYNTH_IMAGES:
        break

    d = coco_data[coco_id]
    gt_caption = d["gt_captions"][0]  # Use first GT caption

    # Step 1: Extract triples from GT caption to find corruptible relations
    gt_triples = stage1_extract_triples(gt_caption)
    if not gt_triples:
        synth["skipped"] += 1
        continue

    # Pick a random triple with a clear supporting_span
    candidates = [t for t in gt_triples if t.get("supporting_span")]
    if not candidates:
        candidates = gt_triples
    target_triple = random.choice(candidates)

    # Step 2: Use LLM to corrupt the relation
    prompt = CORRUPT_PROMPT.format(
        caption=gt_caption,
        subject=target_triple.get("subject", ""),
        relation=target_triple.get("relation", ""),
        object=target_triple.get("object", ""),
        span=target_triple.get("supporting_span", gt_caption),
    )
    corruption = llm_call(prompt, max_tokens=300)
    if not corruption or "corrupted_caption" not in corruption:
        synth["skipped"] += 1
        continue

    corrupted_caption = corruption["corrupted_caption"]

    # Sanity: corrupted caption should differ from GT
    if corrupted_caption.strip() == gt_caption.strip():
        synth["skipped"] += 1
        continue

    # Step 3: Load image and run pipeline on corrupted caption
    try:
        img = Image.open(d["path"]).convert("RGB")
    except Exception:
        synth["skipped"] += 1
        continue

    pipe_result = run_pipeline_on_caption(img, corrupted_caption)

    # Step 4: Measure detection — did we flag the corrupted triple?
    corrupted_rel = corruption.get("corrupted_relation", "").lower()
    original_rel  = corruption.get("original_relation", "").lower()
    subj_lower    = target_triple.get("subject", "").lower()
    obj_lower     = target_triple.get("object", "").lower()

    detected = False
    for v in pipe_result["verdicts"]:
        vt = v["triple"]
        # Match: same subject+object, and relation matches the corrupted one
        if (vt.get("subject", "").lower() == subj_lower and
            vt.get("object", "").lower() == obj_lower and
            v["verdict"].get("verdict") == "unsupported"):
            detected = True
            break

    # Step 5: Measure correction quality
    corrected = pipe_result["corrected_caption"]
    bleu_scores  = compute_bleu(gt_caption, corrected)
    rouge_scores = compute_rouge(gt_caption, corrected)

    # Also measure corrupted-vs-GT as baseline
    bleu_corrupted  = compute_bleu(gt_caption, corrupted_caption)
    rouge_corrupted = compute_rouge(gt_caption, corrupted_caption)

    # Did correction improve over corrupted? (closer to GT)
    correction_improved = bleu_scores["bleu4"] > bleu_corrupted["bleu4"]

    # Did corrected caption remove the corrupted span?
    corrupted_span = corruption.get("corrupted_span", "")
    span_removed = corrupted_span and corrupted_span not in corrected

    synth["corrupted"].append({
        "coco_id": coco_id,
        "gt_caption": gt_caption,
        "gt_captions_all": d["gt_captions"],
        "corrupted_caption": corrupted_caption,
        "corrected_caption": corrected,
        "target_triple": target_triple,
        "corruption_info": corruption,
        "detected": detected,
        "span_removed": span_removed,
        "correction_improved": correction_improved,
        "n_corrections": len(pipe_result["corrections"]),
        "n_triples": len(pipe_result["triples"]),
        "n_unsupported": sum(1 for v in pipe_result["verdicts"]
                            if v["verdict"].get("verdict") == "unsupported"),
        "bleu_corrected": bleu_scores,
        "rouge_corrected": rouge_scores,
        "bleu_corrupted": bleu_corrupted,
        "rouge_corrupted": rouge_corrupted,
    })

    if (len(synth["corrupted"])) % 10 == 0:
        save_ckpt(synth, SYNTH_CKPT)
        n_done = len(synth["corrupted"])
        n_det = sum(1 for r in synth["corrupted"] if r["detected"])
        print(f"  [{n_done}/{N_SYNTH_IMAGES}] Detection recall so far: "
              f"{n_det}/{n_done} = {n_det/n_done:.1%}")


# ══════════════════════════════════════════════════════════
# TRACK B: No-change (clean GT captions → should NOT be modified)
# ══════════════════════════════════════════════════════════
print()
print("=" * 70)
print("TRACK B: Clean GT captions (no-change test)")
print("=" * 70)

already_done_nc = {r["coco_id"] for r in synth["no_change"]}
n_target_nc = max(0, N_SYNTH_IMAGES // 2 - len(synth["no_change"]))
# Use the SECOND half of coco_data for no-change (no overlap with corrupted)
corrupted_ids = {r["coco_id"] for r in synth["corrupted"]}
nc_candidates = [cid for cid in coco_data
                 if cid not in corrupted_ids and cid not in already_done_nc]

print(f"  Already done: {len(synth['no_change'])}, target: {N_SYNTH_IMAGES // 2}")

for idx, coco_id in enumerate(nc_candidates):
    if len(synth["no_change"]) >= N_SYNTH_IMAGES // 2:
        break

    d = coco_data[coco_id]
    gt_caption = d["gt_captions"][0]

    try:
        img = Image.open(d["path"]).convert("RGB")
    except Exception:
        continue

    pipe_result = run_pipeline_on_caption(img, gt_caption)

    corrected = pipe_result["corrected_caption"]
    unchanged = corrected.strip() == gt_caption.strip()
    n_false_pos = sum(1 for v in pipe_result["verdicts"]
                      if v["verdict"].get("verdict") == "unsupported")

    synth["no_change"].append({
        "coco_id": coco_id,
        "gt_caption": gt_caption,
        "corrected_caption": corrected,
        "unchanged": unchanged,
        "n_false_positives": n_false_pos,
        "n_triples": len(pipe_result["triples"]),
        "n_corrections": len(pipe_result["corrections"]),
    })

    if (len(synth["no_change"])) % 10 == 0:
        save_ckpt(synth, SYNTH_CKPT)
        n_done = len(synth["no_change"])
        n_ok = sum(1 for r in synth["no_change"] if r["unchanged"])
        print(f"  [{n_done}/{N_SYNTH_IMAGES // 2}] No-change accuracy: "
              f"{n_ok}/{n_done} = {n_ok/n_done:.1%}")


# ══════════════════════════════════════════════════════════
# SUMMARY — Kim-style Table 2 equivalent
# ══════════════════════════════════════════════════════════
save_ckpt(synth, SYNTH_CKPT)

corrupted_results = synth["corrupted"]
nc_results = synth["no_change"]

n_c = len(corrupted_results)
n_nc = len(nc_results)

if n_c > 0:
    detection_recall   = sum(1 for r in corrupted_results if r["detected"]) / n_c
    span_removal_rate  = sum(1 for r in corrupted_results if r["span_removed"]) / n_c
    correction_improves = sum(1 for r in corrupted_results if r["correction_improved"]) / n_c

    avg_bleu4_corrected = np.mean([r["bleu_corrected"]["bleu4"] for r in corrupted_results])
    avg_bleu4_corrupted = np.mean([r["bleu_corrupted"]["bleu4"] for r in corrupted_results])
    avg_bleu1_corrected = np.mean([r["bleu_corrected"]["bleu1"] for r in corrupted_results])
    avg_bleu1_corrupted = np.mean([r["bleu_corrupted"]["bleu1"] for r in corrupted_results])

    avg_rougeL_corrected = np.mean([r["rouge_corrected"]["rougeL"] for r in corrupted_results])
    avg_rougeL_corrupted = np.mean([r["rouge_corrupted"]["rougeL"] for r in corrupted_results])
    avg_rouge1_corrected = np.mean([r["rouge_corrected"]["rouge1"] for r in corrupted_results])
    avg_rouge1_corrupted = np.mean([r["rouge_corrupted"]["rouge1"] for r in corrupted_results])
else:
    detection_recall = span_removal_rate = correction_improves = 0
    avg_bleu4_corrected = avg_bleu4_corrupted = 0
    avg_bleu1_corrected = avg_bleu1_corrupted = 0
    avg_rougeL_corrected = avg_rougeL_corrupted = 0
    avg_rouge1_corrected = avg_rouge1_corrupted = 0

if n_nc > 0:
    no_change_accuracy = sum(1 for r in nc_results if r["unchanged"]) / n_nc
    avg_false_pos = np.mean([r["n_false_positives"] for r in nc_results])
else:
    no_change_accuracy = avg_false_pos = 0

print("\n" + "=" * 70)
print("SYNTHETIC EVALUATION RESULTS (Kim-style)")
print("=" * 70)

print(f"\n{'Metric':<35} {'Value':>10}")
print("-" * 50)
print(f"{'Corrupted test cases':<35} {n_c:>10}")
print(f"{'No-change test cases':<35} {n_nc:>10}")
print(f"{'Skipped (no triples/corruption)':<35} {synth['skipped']:>10}")

print(f"\n── Track A: Corrupted Captions ──")
print(f"{'Detection recall':<35} {detection_recall:>10.1%}")
print(f"{'Span removal rate':<35} {span_removal_rate:>10.1%}")
print(f"{'Correction improves BLEU-4':<35} {correction_improves:>10.1%}")

print(f"\n── BLEU (vs GT caption) ──")
print(f"{'  BLEU-1 corrupted (baseline)':<35} {avg_bleu1_corrupted:>10.3f}")
print(f"{'  BLEU-1 corrected (RelCheck)':<35} {avg_bleu1_corrected:>10.3f}")
print(f"{'  BLEU-1 delta':<35} {avg_bleu1_corrected - avg_bleu1_corrupted:>+10.3f}")
print(f"{'  BLEU-4 corrupted (baseline)':<35} {avg_bleu4_corrupted:>10.3f}")
print(f"{'  BLEU-4 corrected (RelCheck)':<35} {avg_bleu4_corrected:>10.3f}")
print(f"{'  BLEU-4 delta':<35} {avg_bleu4_corrected - avg_bleu4_corrupted:>+10.3f}")

print(f"\n── ROUGE (vs GT caption) ──")
print(f"{'  ROUGE-1 corrupted (baseline)':<35} {avg_rouge1_corrupted:>10.3f}")
print(f"{'  ROUGE-1 corrected (RelCheck)':<35} {avg_rouge1_corrected:>10.3f}")
print(f"{'  ROUGE-1 delta':<35} {avg_rouge1_corrected - avg_rouge1_corrupted:>+10.3f}")
print(f"{'  ROUGE-L corrupted (baseline)':<35} {avg_rougeL_corrupted:>10.3f}")
print(f"{'  ROUGE-L corrected (RelCheck)':<35} {avg_rougeL_corrected:>10.3f}")
print(f"{'  ROUGE-L delta':<35} {avg_rougeL_corrected - avg_rougeL_corrupted:>+10.3f}")

print(f"\n── Track B: No-Change Accuracy ──")
print(f"{'No-change accuracy':<35} {no_change_accuracy:>10.1%}")
print(f"{'Avg false positives per image':<35} {avg_false_pos:>10.2f}")

# ── Per-corruption-type breakdown ─────────────────────────
if n_c > 0:
    print(f"\n── Detection Recall by Relation Type ──")
    type_counts = defaultdict(lambda: {"total": 0, "detected": 0})
    for r in corrupted_results:
        rtype = r["target_triple"].get("relation_type", "other")
        type_counts[rtype]["total"] += 1
        if r["detected"]:
            type_counts[rtype]["detected"] += 1
    for rtype, counts in sorted(type_counts.items()):
        recall = counts["detected"] / counts["total"] if counts["total"] > 0 else 0
        print(f"  {rtype:<15} {counts['detected']}/{counts['total']} = {recall:.1%}")

# ── Qualitative examples ─────────────────────────────────
print(f"\n── Qualitative Examples (first 5 corrupted) ──")
for r in corrupted_results[:5]:
    status = "DETECTED" if r["detected"] else "MISSED"
    print(f"\n  [{status}] COCO {r['coco_id']}")
    print(f"    GT:        {r['gt_caption'][:100]}")
    print(f"    Corrupted: {r['corrupted_caption'][:100]}")
    print(f"    Corrected: {r['corrected_caption'][:100]}")
    ci = r["corruption_info"]
    print(f"    Changed:   '{ci.get('original_span','')}' → '{ci.get('corrupted_span','')}'")
    print(f"    BLEU-4:    corrupted={r['bleu_corrupted']['bleu4']:.3f} → "
          f"corrected={r['bleu_corrected']['bleu4']:.3f}")

print(f"\nSaved to {SYNTH_CKPT}")